In [32]:
import torch
import torchvision.datasets as datasets
from torchvision.transforms import v2
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer

import numpy as np
import scipy 
import matplotlib.pyplot as plt

import time 
import tqdm
import os

In [223]:
if torch.cuda.is_available():
    device = 'cuda'
else:
    device= 'cpu'
print(device)    

cuda


In [224]:
transforms = v2.Compose([
    v2.RandomResizedCrop(size=(224, 224), antialias=True),
    v2.RandomPhotometricDistort(p=1),
    v2.RandomHorizontalFlip(p=1),
    v2.ToImage()
])
data_train=datasets.Imagenette(os.getcwd()+'\\Desktop',transform=transforms, split = 'train')
data_test=datasets.Imagenette(os.getcwd()+'\\Desktop',transform=transforms, split = 'val')

In [225]:
class ImageDataset(Dataset):
    def __init__(self,data):
        super().__init__()
        self.data=data
        self.transforms=transforms
    def __len__(self):
        return len(self.data)
    def __getitem__(self,index):
        data=self.data
        image=data[index][0].float()
        label=data[index][1]

        if self.transforms:
            image = self.transforms(image)
        return image, label




In [226]:
datatrain = ImageDataset(data = data_train)
datatest = ImageDataset(data = data_test)

In [227]:
batch_size= 2
epochs= 1
lr=10**-1

In [228]:
train_loader = DataLoader(datatrain, batch_size = batch_size)
test_loader = DataLoader(datatest, batch_size = batch_size)

In [229]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*53**2, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net().to(device)

In [230]:
next(net.parameters()).is_cuda

True

In [232]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=lr)

In [233]:
for epoch in range(epochs): 

    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 20 == 19:    
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

[1,    20] loss: 0.002
[1,    40] loss: 0.000
[1,    60] loss: 0.000
[1,    80] loss: 0.000
[1,   100] loss: 0.000
[1,   120] loss: 0.000
[1,   140] loss: 0.000
[1,   160] loss: 0.000
[1,   180] loss: 0.000
[1,   200] loss: 0.000
[1,   220] loss: 0.000
[1,   240] loss: 0.000
[1,   260] loss: 0.000
[1,   280] loss: 0.000
[1,   300] loss: 0.000
[1,   320] loss: 0.000
[1,   340] loss: 0.000
[1,   360] loss: 0.000
[1,   380] loss: 0.000
[1,   400] loss: 0.000
[1,   420] loss: 0.000
[1,   440] loss: 0.000
[1,   460] loss: 0.000
[1,   480] loss: 0.000
[1,   500] loss: 0.198
[1,   520] loss: 0.000
[1,   540] loss: 0.000
[1,   560] loss: 0.000
[1,   580] loss: 0.000
[1,   600] loss: 0.000
[1,   620] loss: 0.000
[1,   640] loss: 0.000
[1,   660] loss: 0.000
[1,   680] loss: 0.000
[1,   700] loss: 0.000
[1,   720] loss: 0.000
[1,   740] loss: 0.000
[1,   760] loss: 0.000
[1,   780] loss: 0.000
[1,   800] loss: 0.000
[1,   820] loss: 0.000
[1,   840] loss: 0.000
[1,   860] loss: 0.000
[1,   880] 

In [ ]:
net(next(iter(train_loader))[0])

In [208]:
len(train_loader)

4735

In [252]:
correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in test_loader:
        images, labels = data
       # images, labels = images.to(device), labels.to(device)
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

Accuracy of the network on the 10000 test images: 9 %
